# Robot Telemetry Dashboard

Choose a robot and one axis, then start live playback at two-second intervals. Pause, continue, or restart playback. Use the range slider to select an inclusive time segment and confirm it to see that axis?s sample count, Q1, median, and Q3.

In [9]:
import asyncio
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.colors import hex_to_rgb, qualitative

project_root = Path.cwd().resolve()
while not (project_root / 'data' / 'raw').is_dir() and project_root != project_root.parent:
    project_root = project_root.parent
raw_data_dir = project_root / 'data' / 'raw'
robot_files = {'ROBOT-001': 'robot_1.csv', 'ROBOT-002': 'robot_2.csv', 'ROBOT-003': 'robot_3.csv'}
robot_series = {}
robot_axes = {}
for robot_id, filename in robot_files.items():
    frame = pd.read_csv(raw_data_dir / filename)
    frame['Time'] = pd.to_datetime(frame['Time'], utc=True)
    axes = [col for col in frame.columns if col.startswith('Axis #') and frame[col].notna().any()]
    axes.sort(key=lambda col: int(col.split('#')[1]))
    robot_axes[robot_id] = axes
    robot_series[robot_id] = frame.set_index('Time')[axes].apply(pd.to_numeric, errors='coerce').resample('2s').mean().dropna(how='all')

axis_colors = qualitative.Dark24
visible_window_points = 300
progress = {robot_id: 0 for robot_id in robot_files}
playback_state = {'playing': False, 'generation': 0}

figure = go.FigureWidget()
figure.add_scatter(x=[], y=[], mode='lines', name='Axis #1', fill='tozeroy')
figure.update_layout(
    title='ROBOT-001 Current by Axis', xaxis_title='Time (UTC)',
    yaxis_title='Current (Amps)', yaxis={'rangemode': 'tozero'},
    hovermode='x unified', template='plotly_white', height=620,
    margin={'t': 45, 'r': 30, 'b': 60, 'l': 70},
)
robot_selector = widgets.Dropdown(options=list(robot_files), value='ROBOT-001', description='Robot:', layout={'width': '220px'})
axis_selector = widgets.Dropdown(options=robot_axes['ROBOT-001'], value=robot_axes['ROBOT-001'][0], description='Axis:', layout={'width': '220px'})
start_button = widgets.Button(description='Start', icon='play', button_style='success', tooltip='Start the selected robot and axis')
pause_button = widgets.Button(description='Pause', icon='pause', tooltip='Pause playback and keep the current position', disabled=True)
continue_button = widgets.Button(description='Continue', icon='play', tooltip='Continue from the selected robot?s saved position', disabled=True)
restart_button = widgets.Button(description='Restart', icon='refresh', tooltip='Restart the selected robot from its first sample')
status = widgets.HTML(value='Choose a robot and axis, then press Start.')

segment_slider = widgets.IntRangeSlider(
    value=(0, len(robot_series['ROBOT-001']) - 1),
    min=0,
    max=len(robot_series['ROBOT-001']) - 1,
    step=1,
    description='Time segment',
    continuous_update=False,
    readout_format='d',
    style={'description_width': 'initial'},
    layout={'width': '100%'},
)
segment_label = widgets.HTML()
confirm_segment_button = widgets.Button(
    description='Confirm segment', icon='check', button_style='primary',
    tooltip='Calculate quartiles for the selected axis and time segment',
)
quartile_output = widgets.HTML()


def update_segment_label(*_):
    samples = robot_series[robot_selector.value]
    first, last = segment_slider.value
    start_time = samples.index[first].strftime('%Y-%m-%d %H:%M:%S UTC')
    end_time = samples.index[last].strftime('%Y-%m-%d %H:%M:%S UTC')
    segment_label.value = f'{start_time} to {end_time} ({last - first + 1:,} two-second samples)'


def update_figure():
    robot_id = robot_selector.value
    axis = axis_selector.value
    samples = robot_series[robot_id]
    cursor = progress[robot_id]
    visible = samples.iloc[max(0, cursor - visible_window_points):cursor][axis].dropna()
    color = axis_colors[int(axis.split('#')[1]) - 1]
    red, green, blue = hex_to_rgb(color)
    with figure.batch_update():
        trace = figure.data[0]
        trace.x = visible.index
        trace.y = visible.to_numpy()
        trace.name = axis
        trace.line.color = color
        trace.fillcolor = f'rgba({red}, {green}, {blue}, 0.12)'
        figure.layout.title = f'{robot_id} {axis} Current'
        if len(visible) > 1:
            figure.layout.xaxis.range = [visible.index[0], visible.index[-1]]
            figure.layout.xaxis.autorange = False
        else:
            figure.layout.xaxis.autorange = True


async def play_robot(robot_id, generation):
    samples = robot_series[robot_id]
    while playback_state['playing'] and playback_state['generation'] == generation and robot_selector.value == robot_id:
        cursor = progress[robot_id]
        if cursor >= len(samples):
            break
        progress[robot_id] = cursor + 1
        if robot_selector.value == robot_id:
            update_figure()
        stamp = samples.index[cursor].strftime('%Y-%m-%d %H:%M:%S UTC')
        status.value = f'Playing {robot_id}: sample {cursor + 1:,} of {len(samples):,} | {stamp}'
        if progress[robot_id] >= len(samples):
            break
        await asyncio.sleep(2)
    if playback_state['generation'] == generation:
        playback_state['playing'] = False
        pause_button.disabled = True
        continue_button.disabled = progress[robot_id] >= len(samples)
        if progress[robot_id] >= len(samples):
            status.value = f'{robot_id} playback complete. Press Restart to replay.'


def start_playback(_=None):
    if progress[robot_selector.value] != 0:
        status.value = 'This robot has already started. Use Continue or Restart.'
        return
    continue_playback()


def continue_playback(_=None):
    if playback_state['playing']:
        return
    robot_id = robot_selector.value
    if progress[robot_id] >= len(robot_series[robot_id]):
        status.value = f'{robot_id} is complete. Press Restart to replay.'
        return
    playback_state['generation'] += 1
    generation = playback_state['generation']
    playback_state['playing'] = True
    start_button.disabled = True
    continue_button.disabled = True
    pause_button.disabled = False
    asyncio.create_task(play_robot(robot_id, generation))


def pause_playback(_=None):
    if not playback_state['playing']:
        return
    robot_id = robot_selector.value
    playback_state['playing'] = False
    playback_state['generation'] += 1
    pause_button.disabled = True
    continue_button.disabled = progress[robot_id] >= len(robot_series[robot_id])
    status.value = f'{robot_id} paused at sample {progress[robot_id]:,} of {len(robot_series[robot_id]):,}.'


def restart_playback(_=None):
    robot_id = robot_selector.value
    playback_state['playing'] = False
    playback_state['generation'] += 1
    progress[robot_id] = 0
    start_button.disabled = False
    continue_button.disabled = True
    pause_button.disabled = True
    update_figure()
    status.value = f'Restarting {robot_id} from its first sample.'
    continue_playback()


def on_robot_change(change):
    if change['name'] != 'value':
        return
    playback_state['playing'] = False
    playback_state['generation'] += 1
    robot_id = change['new']
    available_axes = robot_axes[robot_id]
    axis_selector.options = available_axes
    if axis_selector.value not in available_axes:
        axis_selector.value = available_axes[0]
    cursor = progress[robot_id]
    start_button.disabled = cursor > 0
    continue_button.disabled = cursor == 0 or cursor >= len(robot_series[robot_id])
    pause_button.disabled = True
    status.value = f'{robot_id} paused at sample {cursor:,} of {len(robot_series[robot_id]):,}. Press Start or Continue.'
    maximum = len(robot_series[robot_id]) - 1
    segment_slider.value = (0, 0)
    segment_slider.max = maximum
    segment_slider.value = (0, maximum)
    quartile_output.value = ''
    update_figure()
    update_segment_label()


def on_axis_change(change):
    if change['name'] == 'value':
        update_figure()
        quartile_output.value = ''


def confirm_segment(_=None):
    robot_id = robot_selector.value
    axis = axis_selector.value
    first, last = segment_slider.value
    segment = robot_series[robot_id][axis].iloc[first:last + 1].dropna()
    if segment.empty:
        quartile_output.value = '<strong>No telemetry samples in the selected segment.</strong>'
        return
    statistics = segment.describe(percentiles=[0.25, 0.5, 0.75])
    quartiles = pd.DataFrame([{
        'Samples': int(statistics['count']),
        'Q1 (25%)': statistics['25%'],
        'Median (50%)': statistics['50%'],
        'Q3 (75%)': statistics['75%'],
    }], index=[axis])
    quartiles.index.name = 'Axis'
    table_html = quartiles.round(3).to_html(index=True, border=0, classes='quartile-table')
    start_time = segment.index[0].strftime('%Y-%m-%d %H:%M:%S UTC')
    end_time = segment.index[-1].strftime('%Y-%m-%d %H:%M:%S UTC')
    quartile_output.value = (
        f'<strong>{robot_id} | {axis}</strong> | {start_time} to {end_time} '
        f'| {len(segment):,} two-second samples<div>{table_html}</div>'
    )


robot_selector.observe(on_robot_change, names='value')
axis_selector.observe(on_axis_change, names='value')
segment_slider.observe(update_segment_label, names='value')
start_button.on_click(start_playback)
pause_button.on_click(pause_playback)
continue_button.on_click(continue_playback)
restart_button.on_click(restart_playback)
confirm_segment_button.on_click(confirm_segment)
update_figure()
update_segment_label()
controls = widgets.VBox([
    widgets.HBox([robot_selector, axis_selector]),
    widgets.HBox([start_button, pause_button, continue_button, restart_button]),
    status,
    segment_slider,
    segment_label,
    confirm_segment_button,
    quartile_output,
])
display(widgets.VBox([controls, figure]))
